In [115]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_validate, GridSearchCV, RandomizedSearchCV
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib

In [78]:
df = pd.read_csv('Datasets\cumulative_cleaned.csv')
df.sample(5)

,koi_period,koi_time0bk,koi_impact,koi_duration,koi_depth,koi_prad,koi_teq,koi_insol,koi_model_snr,koi_steff,koi_slogg,koi_srad,koi_kepmag,koi_disposition
2570,13.981779,181.975114,0.444,2.95800,1018.4,2.09,553.0,22.10,95.6,4859.0,4.615,0.651,14.287,1
5303,2.305903,131.792859,0.003,3.38280,23269.0,39.04,3147.0,23214.46,511.7,9696.0,4.033,2.492,16.321,0
621,0.576079,134.107265,1.113,1.63253,10085.0,23.23,1982.0,3647.63,426.0,5693.0,4.570,0.816,15.237,0
920,2.617028,134.032940,0.146,2.04600,499.4,1.57,996.0,233.23,30.8,4831.0,4.568,0.724,15.835,1
3994,2.080432,131.827995,0.771,7.49970,894200.0,345.70,2719.0,12956.98,1585.2,8344.0,4.040,2.115,16.448,0


In [79]:
features = ['koi_period', 'koi_time0bk',  'koi_impact', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_model_snr', 'koi_steff', 'koi_slogg',  'koi_srad','koi_kepmag']

In [80]:
X = df[features]
y = df['koi_disposition']

In [81]:
X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [82]:
skewed_cols = ['koi_period', 'koi_time0bk',  'koi_impact', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_model_snr', 'koi_steff', 'koi_slogg',  'koi_srad','koi_kepmag']

In [83]:
# log transformation
X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
X_test[skewed_cols] = np.log1p(X_test[skewed_cols])

In [84]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) 
X_test = scaler.transform(X_test)   

In [91]:
log_reg_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000, class_weight='balanced'))
])

log_reg_pipeline.fit(X_train, y_train)

y_pred_logreg = log_reg_pipeline.predict(X_test)
y_proba_logreg = log_reg_pipeline.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_logreg))

              precision    recall  f1-score   support

           0       0.94      0.79      0.86       945
           1       0.68      0.89      0.77       458

    accuracy                           0.82      1403
   macro avg       0.81      0.84      0.81      1403
weighted avg       0.85      0.82      0.83      1403



In [92]:
rf_pipeline = Pipeline(steps=[
    ('model', RandomForestClassifier(n_estimators= 10, random_state=42, class_weight='balanced'))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.93      0.93      0.93       945
           1       0.85      0.86      0.86       458

    accuracy                           0.91      1403
   macro avg       0.89      0.89      0.89      1403
weighted avg       0.91      0.91      0.91      1403



In [93]:
from sklearn.ensemble import GradientBoostingClassifier

gb_pipeline = Pipeline(steps=[
    ('model', GradientBoostingClassifier(random_state=42))
])

gb_pipeline.fit(X_train, y_train)

y_pred_gb = gb_pipeline.predict(X_test)
y_proba_gb = gb_pipeline.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_gb))

              precision    recall  f1-score   support

           0       0.95      0.92      0.94       945
           1       0.85      0.90      0.88       458

    accuracy                           0.92      1403
   macro avg       0.90      0.91      0.91      1403
weighted avg       0.92      0.92      0.92      1403



In [102]:
svm_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf', class_weight='balanced', random_state=42 ))
])

svm_pipeline.fit(X_train, y_train)

y_pred_svm = svm_pipeline.predict(X_test)
y_proba_svm = svm_pipeline.decision_function(X_test)

print(classification_report(y_test, y_pred_svm))

              precision    recall  f1-score   support

           0       0.98      0.89      0.93       945
           1       0.80      0.95      0.87       458

    accuracy                           0.91      1403
   macro avg       0.89      0.92      0.90      1403
weighted avg       0.92      0.91      0.91      1403



In [103]:
results = []

models = {
    'Logistic Regression': (y_pred_logreg, y_proba_logreg),
    'Random Forest': (y_pred_rf, y_proba_rf),
    'Gradient Boosting': (y_pred_gb, y_proba_gb),
    'SVM': (y_pred_svm, y_proba_svm)
}

for name, (pred, proba) in models.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision Macro': precision_score(y_test, pred, average='macro', zero_division=0),
        'Recall Macro': recall_score(y_test, pred, average='macro', zero_division=0),
        'F1 Macro': f1_score(y_test, pred, average='macro'),
        'ROC_AUC': roc_auc_score(y_test, proba)
    })

results_df = pd.DataFrame(results).sort_values(by='F1 Macro', ascending=False)
results_df

,Model,Accuracy,Precision Macro,Recall Macro,F1 Macro,ROC_AUC
2,Gradient Boosting,0.917320,0.902170,0.913307,0.907313,0.966002
3,SVM,0.908767,0.889431,0.920460,0.900651,0.965939
1,Random Forest,0.905203,0.891149,0.894186,0.892636,0.959820
0,Logistic Regression,0.823949,0.805351,0.840057,0.812688,0.917245


In [104]:
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc']

pipelines = {
    'Logistic Regression': log_reg_pipeline,
    'Random Forest': rf_pipeline,
    'Gradient Boosting': gb_pipeline,
    'SVM': svm_pipeline
}

cv_results = []

for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X, y, cv=5, scoring=scoring)

    cv_results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision Macro': scores['test_precision_macro'].mean(),
        'Recall Macro': scores['test_recall_macro'].mean(),
        'F1 Macro': scores['test_f1_macro'].mean(),
        'ROC_AUC': scores['test_roc_auc'].mean()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(by='F1 Macro', ascending=False)

cv_results_df

,Model,Accuracy,Precision Macro,Recall Macro,F1 Macro,ROC_AUC
2,Gradient Boosting,0.899359,0.901200,0.890404,0.887031,0.969086
1,Random Forest,0.895225,0.895663,0.882841,0.880826,0.956882
3,SVM,0.816108,0.813298,0.841164,0.807254,0.916756
0,Logistic Regression,0.796721,0.793392,0.823744,0.787755,0.889092


In [116]:
gb_tuning_pipeline = Pipeline(steps=[
    ('model', GradientBoostingClassifier(
        random_state=42,))
])

param_dist = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'model__max_depth': [2, 3, 4, 5],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__subsample': [0.6, 0.8, 1.0],
    'model__max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    estimator=gb_tuning_pipeline,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1_macro',      # since you're comparing by F1 Macro
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

total_fits = random_search.n_iter * random_search.cv

with tqdm_joblib(tqdm(desc="RandomizedSearchCV", total=total_fits)):
    random_search.fit(X_train, y_train)

print(random_search.best_params_)
print(random_search.best_score_)

RandomizedSearchCV:   0%|          | 0/250 [00:00<?, ?it/s]

Fitting 5 folds for each of 50 candidates, totalling 250 fits


100%|██████████| 250/250 [02:31<00:00,  1.65it/s]

{'model__subsample': 0.8, 'model__n_estimators': 300, 'model__min_samples_split': 5, 'model__min_samples_leaf': 4, 'model__max_features': None, 'model__max_depth': 4, 'model__learning_rate': 0.1}
0.9223175228430683


In [117]:
#Evaluate tuned Gradient Boosting on the test set

best_gb_pipeline = random_search.best_estimator_

y_pred_best_gb = best_gb_pipeline.predict(X_test)
y_proba_best_gb = best_gb_pipeline.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_best_gb))
print('ROc-AUC:', roc_auc_score(y_test, y_proba_best_gb))

              precision    recall  f1-score   support

           0       0.96      0.93      0.95       945
           1       0.87      0.92      0.89       458

    accuracy                           0.93      1403
   macro avg       0.92      0.93      0.92      1403
weighted avg       0.93      0.93      0.93      1403

ROc-AUC: 0.9706014186363533


In [119]:
#Add tuned Gradient Boosting to model comparison

tuned_gb_result = pd.DataFrame([{
    'Model': 'Tuned Gradient Boosting',
    'Accuracy': accuracy_score(y_test, y_pred_best_gb),
    'Precision Macro': precision_score(y_test, y_pred_best_gb, average='macro', zero_division=0),
    'Recall Macro': recall_score(y_test, y_pred_best_gb, average='macro', zero_division=0),
    'F1 Macro': f1_score(y_test, y_pred_best_gb, average='macro'),
    'ROC_AUC': roc_auc_score(y_test, y_proba_best_gb)
}])

final_results_df = pd.concat([results_df, tuned_gb_result], ignore_index=True)
final_results_df = final_results_df.sort_values(by='F1 Macro', ascending=False)

final_results_df

,Model,Accuracy,Precision Macro,Recall Macro,F1 Macro,ROC_AUC
4,Tuned Gradient Boosting,0.929437,0.915709,0.926803,0.920855,0.970601
0,Gradient Boosting,0.917320,0.902170,0.913307,0.907313,0.966002
1,SVM,0.908767,0.889431,0.920460,0.900651,0.965939
2,Random Forest,0.905203,0.891149,0.894186,0.892636,0.959820
3,Logistic Regression,0.823949,0.805351,0.840057,0.812688,0.917245


In [120]:
#Select final model automatically from the comparison table

model_objects = {
    'Logistic Regression': (log_reg_pipeline, y_pred_logreg, y_proba_logreg),
    'Random Forest': (rf_pipeline, y_pred_rf, y_proba_rf),
    'Gradient Boosting': (gb_pipeline, y_pred_gb, y_proba_gb),
    'SVM': (svm_pipeline, y_pred_svm, y_proba_svm),
    'Tuned Gradient Boosting': (best_gb_pipeline, y_pred_best_gb, y_proba_best_gb)
}

best_model_name = final_results_df.sort_values(
    by=['F1 Macro', 'ROC_AUC'],
    ascending=False
).iloc[0]['Model']

best_pipeline, y_pred_best, y_proba_best = model_objects[best_model_name]

print("Final selected model:", best_model_name)
print(final_results_df[final_results_df['Model'] == best_model_name])


Final selected model: Tuned Gradient Boosting
                     Model  Accuracy  Precision Macro  Recall Macro  F1 Macro  \
4  Tuned Gradient Boosting  0.929437         0.915709      0.926803  0.920855   

    ROC_AUC  
4  0.970601  
